In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
import pandas as pd 
import matplotlib as mt 
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import AutoTokenizer
import pickle as pk

/mnt/storage-drive/code_project/resuming/ai-ml/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset

In [ ]:
pd_dataset = pd.read_json('dataset/dataset.json')

In [ ]:
features = [item['sentence'] for item in pd_dataset.to_dict('records')]

In [5]:
labels = [item['skills'] for item in pd_dataset.to_dict('records')]

In [6]:
mlb = MultiLabelBinarizer()

In [7]:
binary_labels = mlb.fit_transform(labels)

In [8]:
with open('skill_encoder.pkl', 'wb') as file:
    pk.dump(mlb, file)

## Transformer

In [9]:
tokeniser = AutoTokenizer.from_pretrained('google-bert/bert-base-cased')

## Dataset Class

In [10]:
class SkillExtractionDataset(Dataset):
    def __init__(self, feature, label, tokeniser):
        self.feature = feature
        self.label = label
        self.tokeniser = tokeniser

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        current_feature = self.feature[idx]
        current_label = self.label[idx]

        tokenised_feature = self.tokeniser(
            text=current_feature,
            padding="max_length",
            truncation=True,
            max_length=100,
            return_tensors="pt",
        )

        input_ids = tokenised_feature["input_ids"].squeeze(0)
        label_tensor = torch.tensor(current_label, dtype=torch.float32)

        return input_ids, label_tensor

In [11]:
pt_dataset = SkillExtractionDataset(feature=features, label=binary_labels, tokeniser=tokeniser)

In [12]:
dataloader = DataLoader(pt_dataset, batch_size=60, shuffle=True)

## Model

In [13]:
class SkillExtractionModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(35000, 20)
        self.conv1 = nn.Conv1d(100, 300, 3)
        self.conv2 = nn.Conv1d(300, 100, 3)
        self.linear1 = nn.Linear(16, 1)
        self.linear2 = nn.Linear(100, num_classes)

    def forward(self, feature):
        x = self.embedding(feature)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.linear1(x)
        x = x.squeeze(-1)
        x = self.linear2(x)

        return x

In [14]:
model = SkillExtractionModel(num_classes=len(mlb.classes_))

## Training

In [15]:
optimiser = optim.AdamW(model.parameters(), lr=0.001)

In [16]:
criterion = nn.BCEWithLogitsLoss()

In [17]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser, factor=0.8)

In [18]:
for i in range(1000):
    for batch in dataloader:
        feature, label = batch

        prediction = model(feature)

        loss = criterion(prediction, label)

        print(loss)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    
    scheduler.step(loss)

tensor(0.6937, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)


/mnt/storage-drive/code_project/resuming/ai-ml/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


tensor(0.6145, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.5365, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.4488, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.3574, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2728, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2009, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.1516, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.1047, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0749, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)


/mnt/storage-drive/code_project/resuming/ai-ml/.venv/lib/python3.13/site-packages/torch/optim/lr_scheduler.py:1691: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  current = float(metrics)


tensor(0.0682, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0749, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0735, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0831, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0820, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0842, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0721, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0823, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0886, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0681, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0740, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0719, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0618, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0679, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0621, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0578, grad_fn=<BinaryCrossEntro

KeyboardInterrupt: 

In [ ]:
torch.save(model, 'model/model.pth')